In [1]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")
#sys.path.append("/Users/andre/Documents/xerenity/pysdk")
import numpy as np
import numpy_financial as npf
from scipy.optimize import newton
import os
import json
from src.xerenity.xty import Xerenity
from server.loan_calculator.loan_calculator import LoanCalculatorServer
from utilities.date_functions import datetime_to_ql,ql_to_datetime
import pandas as pd
import QuantLib as ql
from loans_calculator.funciones_analisis_credito import 

from utilities.date_functions import days_30_360_dt,days_30_360_ql,days_act_act_dt,days_act_act_ql,days_act_365_ql,days_act_365_dt
xty = Xerenity(
    username=os.getenv('XTY_USER'),
    password=os.getenv('XTY_PWD'),
)

ibr_cashflow = xty.get_ibr_data(
    loan_id="beb65020-e940-4995-9e1e-2f2208cdc638",
    filter_date="2024-08-23"
)
calc = LoanCalculatorServer(ibr_cashflow, local_dev=True)
loan_payments = calc.cash_flow_ibr()


# Define the value_date
value_date = ql.Date(15, 7, 2024)

2024-08-28 20:44:27,292:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"
2024-08-28 20:44:27,930:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data "HTTP/1.1 200 OK"
/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments


In [74]:
loan_payments=pd.DataFrame(loan_payments)
loan_payments

,date,beginning_balance,rate,rate_tot,payment,interest,principal,ending_balance
0,2024-04-06,1.000000e+09,11.735000,16.735000,9.727917e+07,1.394583e+07,8.333333e+07,9.166667e+08
1,2024-05-06,9.166667e+08,11.380000,16.380000,9.584583e+07,1.251250e+07,8.333333e+07,8.333333e+08
2,2024-06-06,8.333333e+08,10.997000,15.997000,9.444236e+07,1.110903e+07,8.333333e+07,7.500000e+08
3,2024-07-06,7.500000e+08,10.919000,15.919000,9.328271e+07,9.949375e+06,8.333333e+07,6.666667e+08
4,2024-08-06,6.666667e+08,10.431000,15.431000,9.190611e+07,8.572778e+06,8.333333e+07,5.833333e+08
5,2024-09-06,5.833333e+08,10.090000,15.090000,9.066875e+07,7.335417e+06,8.333333e+07,5.000000e+08
6,2024-10-06,5.000000e+08,9.985923,14.985923,8.957747e+07,6.244134e+06,8.333333e+07,4.166667e+08
7,2024-11-06,4.166667e+08,9.248855,14.248855,8.828085e+07,4.947519e+06,8.333333e+07,3.333333e+08
8,2024-12-06,3.333333e+08,9.440521,14.440521,8.734459e+07,4.011256e+06,8.333333e+07,2.500000e+08
9,2025-01-06,2.500000e+08,8.402619,13.402619,8.612555e+07,2.792212e+06,8.333333e+07,1.666667e+08


In [63]:
calculate_days_from_value_date(loan_payments,value_date,'30/360')

{'previous_payment_date': Timestamp('2024-07-06 00:00:00'),
 'days_to_previous': -9,
 'next_payment_date': Timestamp('2024-08-06 00:00:00'),
 'days_to_next': 21,
 'next_interest_payment': 8572777.777777776,
 'accrued_interest': 2571833.333333333}

In [64]:
calculate_days_from_value_date(loan_payments,value_date,'actual/365')

{'previous_payment_date': Timestamp('2024-07-06 00:00:00'),
 'days_to_previous': -9,
 'next_payment_date': Timestamp('2024-08-06 00:00:00'),
 'days_to_next': 22,
 'next_interest_payment': 8572777.777777776,
 'accrued_interest': 2488870.967741935}

In [65]:
flows=create_cashflows_and_total_value(loan_payments, value_date,'30/360')

In [66]:
import pandas as pd
import numpy_financial as npf

def calculate_irr(dates, cashflows):
    """
    Calculate the Internal Rate of Return (IRR) for a series of cash flows and their associated dates.

    Parameters:
    - dates (list of pd.Timestamp): List of dates for each cash flow
    - cashflows (list of float): List of cash flows corresponding to each date

    Returns:
    - irr (float): Internal Rate of Return (IRR)
    """
    # Ensure dates and cashflows are in pandas Series
    dates = pd.Series(dates)
    cashflows = pd.Series(cashflows)

    # Ensure that dates are sorted
    sorted_df = pd.DataFrame({'date': dates, 'cashflow': cashflows}).sort_values(by='date')

    # Calculate the time periods in years relative to the first date
    sorted_df['time'] = (sorted_df['date'] - sorted_df['date'].min()) / pd.Timedelta(days=365.25)

    # Extract the cash flows and corresponding time periods
    cashflows = sorted_df['cashflow'].values

    # Compute the IRR using numpy_financial's financial function
    irr = npf.irr(cashflows)
    
    return irr

# Example usage
dates = [
    pd.Timestamp('2024-01-01'),
    pd.Timestamp('2024-06-01'),
    pd.Timestamp('2025-01-01')
]
cashflows = [-1000, 500, 600]  # Example cashflows: initial investment followed by returns

irr = calculate_irr(dates, cashflows)
print(f"IRR: {irr:.4f}")



IRR: 0.0639


In [67]:
flows.to_clipboard()

In [72]:
calculate_irr(flows['date'],flows['payment'],'30/360')*100

17.26210233156504

In [69]:
loan_payments.to_clipboard()